# Week 9: Raster & Remote Sensing

This notebook covers:
- Loading satellite imagery with Rasterio
- Clipping rasters to an area of interest
- Calculating change between two dates
- Zonal statistics

---

## Before you start

You'll need satellite imagery files for this notebook. Make sure you've:
1. Downloaded the Week 9 data from the course data guide
2. Uploaded it to your Google Drive (Colab) or saved it locally (Jupyter)

---

## Step 0: Set up environment

In [ ]:
# Detect environment and install packages
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    print("Installing packages...")
    !pip install geopandas rasterio rasterstats -q
    print("Done!")
else:
    print("Running locally")
    print("Make sure you activated: conda activate intro-gis")

---

## Step 1: Connect to your data

**Colab users:** This mounts your Google Drive so the notebook can access your files.

Your Drive folder should look like:
```
My Drive/
└── intro-gis/
    └── data/
        ├── raw/              ← Input data goes here
        │   ├── aoi.geojson
        │   ├── sentinel_before.tif
        │   ├── sentinel_after.tif
        │   └── zones.geojson
        └── processed/        ← Your outputs save here
```

In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RAW = Path("/content/drive/MyDrive/intro-gis/data/raw")
    PROCESSED = Path("/content/drive/MyDrive/intro-gis/data/processed")
else:
    RAW = Path("../data/raw")
    PROCESSED = Path("../data/processed")

print(f"Reading input data from: {RAW}")
print(f"Saving outputs to: {PROCESSED}")

if RAW.exists():
    print("\nInput folder found! Files:")
    for f in RAW.glob("*"):
        print(f"  {f.name}")
else:
    print("\nInput folder NOT found - check your Drive/local folder structure")

# Create processed folder if it doesn't exist
PROCESSED.mkdir(parents=True, exist_ok=True)
print(f"\nProcessed folder ready: {PROCESSED}")

---

## Step 2: Import libraries

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported!")

---

## Step 3: Load area of interest

The AOI defines the boundary we'll clip our satellite imagery to.

In [ ]:
aoi = gpd.read_file(RAW / "aoi.geojson")

print(f"AOI bounds: {aoi.total_bounds}")
aoi.plot(figsize=(8, 6))
plt.title("Area of Interest")
plt.show()

---

## Step 4: Clip rasters to AOI

Crop the satellite images to your study area (like Clip Raster by Mask Layer in QGIS).

In [ ]:
def clip_raster(raster_path, shapes):
    """Clip a raster to a vector boundary."""
    with rasterio.open(raster_path) as src:
        out_image, out_transform = mask(src, shapes.geometry, crop=True)
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
        })
    return out_image, out_meta

# Clip both images
before, before_meta = clip_raster(RAW / "sentinel_before.tif", aoi)
after, after_meta = clip_raster(RAW / "sentinel_after.tif", aoi)

print(f"Clipped raster shape: {before.shape}")
print(f"  Bands: {before.shape[0]}, Height: {before.shape[1]}, Width: {before.shape[2]}")

---

## Step 5: Calculate change

Compute the difference between the two dates to detect change.

In [ ]:
# Extract first band and convert to float
before_band = before[0].astype(float)
after_band = after[0].astype(float)

# Calculate change
change = after_band - before_band

print(f"Change statistics:")
print(f"  Mean: {np.nanmean(change):.3f}")
print(f"  Min:  {np.nanmin(change):.3f}")
print(f"  Max:  {np.nanmax(change):.3f}")

---

## Step 6: Visualize change

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(before_band, cmap="gray")
axes[0].set_title("Before")
axes[0].axis("off")

axes[1].imshow(after_band, cmap="gray")
axes[1].set_title("After")
axes[1].axis("off")

im = axes[2].imshow(change, cmap="RdYlGn", vmin=-0.3, vmax=0.3)
axes[2].set_title("Change (Green=increase, Red=decrease)")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

---

## Step 7: Zonal statistics

Summarize change values by zone (like QGIS Zonal Statistics tool).

In [ ]:
from rasterstats import zonal_stats

# Load zones
zones = gpd.read_file(RAW / "zones.geojson")

# Calculate zonal statistics
stats = zonal_stats(
    zones, 
    change, 
    affine=before_meta["transform"],
    stats=["mean", "min", "max"],
    nodata=np.nan
)

# Add to GeoDataFrame
zones["change_mean"] = [s["mean"] for s in stats]
zones["change_min"] = [s["min"] for s in stats]
zones["change_max"] = [s["max"] for s in stats]

zones[["name", "change_mean", "change_min", "change_max"]].head()

---

## Step 8: Map zonal results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

zones.plot(
    column="change_mean",
    cmap="RdYlGn",
    legend=True,
    legend_kwds={"label": "Mean Change"},
    ax=ax
)

ax.set_title("Mean Change by Zone")
ax.set_axis_off()
plt.show()

---

## Step 9: Export results

Save the zones with change statistics to the processed folder.

In [ ]:
# Save zones with change statistics to processed folder
zones.to_file(PROCESSED / "zones_change.gpkg", driver="GPKG")

print(f"Saved to: {PROCESSED / 'zones_change.gpkg'}")
print(f"\nYou can now open this file in QGIS from your processed folder!")

---

## Done!

You've completed raster change detection in Python:
1. Loaded and clipped satellite imagery
2. Calculated change between two dates
3. Summarized change by administrative zones
4. Exported results for QGIS

**Save your work:**
- Colab: `File > Save a copy in Drive`
- Local: `Ctrl+S` or `Cmd+S`